# 02 — Feature Selection (PSO)

**Pipeline version:** 2.0-consolidated  
**Purpose:** Two-stage feature selection: Variance Threshold → Mutual Information →
Binary PSO with inner CV fitness & penalty.  
**Inputs:** `X_train_preprocessed.csv`, `y_train.csv` (from NB01)  
**Outputs:** `selected_features_final.csv`, `X_train_selected.csv`, `X_test_selected.csv`  

**Leakage prevention:** Feature selection is performed on the FULL training set
(test set is never seen).  For honest CV estimates, feature selection must be
repeated inside each outer CV fold — see NB03/04 for nested CV.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name != "core" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import config
from src.io import logger, save_dataframe
from src.feature_selection import run_feature_selection, fit_filter_selector, transform_selected

logger.info("Pipeline version: %s", config.PIPELINE_VERSION)

2026-08-26 02:26:45 | INFO     | prostate_bcr | Pipeline version: 2.0-consolidated


## 1. Load Preprocessed Data

In [2]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv")
y_train = pd.read_csv(config.PROCESSED_DIR / "y_train.csv").iloc[:, 0]
X_test = pd.read_csv(config.PROCESSED_DIR / "X_test_preprocessed.csv")
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

logger.info("X_train: %s | y_train: %s (pos=%.1f%%)", X_train.shape, y_train.shape, y_train.mean()*100)
logger.info("X_test:  %s | y_test:  %s (pos=%.1f%%)", X_test.shape, y_test.shape, y_test.mean()*100)

2026-08-26 02:26:48 | INFO     | prostate_bcr | X_train: (343, 19019) | y_train: (343,) (pos=13.4%)
2026-08-26 02:26:48 | INFO     | prostate_bcr | X_test:  (86, 19019) | y_test:  (86,) (pos=14.0%)


## 2. Feature Selection Pipeline: Variance → MI → PSO

In [3]:
fitted_selector, selected_features = run_feature_selection(
    X_train, y_train,
    variance_threshold=config.VARIANCE_THRESHOLD,
    mi_top_k=config.MI_TOP_K,
    pso_final_k=config.PSO_FINAL_K,
    run_pso=True,
)

logger.info("\n" + "="*60)
logger.info("Feature selection complete!")
logger.info("  Variance features: %d", len(fitted_selector["variance_features"]))
logger.info("  MI features: %d", len(fitted_selector["mi_features"]))
logger.info("  Engineered features: %d", len(fitted_selector.get("engineered_features", [])))
logger.info("  Final selected: %d", len(selected_features))
logger.info("  Selected features: %s", selected_features)

2026-08-26 02:27:20 | INFO     | prostate_bcr | Filter selector: 18983 variance → 200 MI features
2026-08-26 02:27:20 | INFO     | prostate_bcr | Engineered: Gleason_Total, High_Risk_Gleason
2026-08-26 02:27:20 | INFO     | prostate_bcr | Engineered: Margin_x_LymphNode
2026-08-26 02:27:20 | INFO     | prostate_bcr | Engineered: T_Stage_Risk
2026-08-26 02:27:20 | INFO     | prostate_bcr | Engineered: PSA_Pathway_Score (from 7 genes)
2026-08-26 02:27:20 | INFO     | prostate_bcr | Engineered: AR_Signaling_Score (from 8 genes)
2026-08-26 02:27:20 | INFO     | prostate_bcr | Engineered: Proliferation_Score (from 7 genes)
2026-08-26 02:27:20 | INFO     | prostate_bcr | Candidate pool: 200 MI features + 7 engineered features = 207 total candidates
2026-08-26 02:27:25 | INFO     | prostate_bcr | PSO: 12 particles, 10 iterations, target 30 features, alpha=0.0010
2026-08-26 02:27:43 | INFO     | prostate_bcr |   PSO iter 5/10: best Fitness=0.8327 (Raw AUC ≈ 0.8627)
2026-08-26 02:28:02 | INFO   

## 3. Apply Selection to Train and Test

In [4]:
# Apply to test using the fitted selector (no leakage)
X_test_selected = transform_selected(X_test, fitted_selector, selected_features)

# Apply to training set
X_train_selected = transform_selected(X_train, fitted_selector, selected_features)

logger.info("X_train_selected: %s", X_train_selected.shape)
logger.info("X_test_selected:  %s", X_test_selected.shape)

2026-08-26 02:28:02 | INFO     | prostate_bcr | X_train_selected: (343, 30)
2026-08-26 02:28:02 | INFO     | prostate_bcr | X_test_selected:  (86, 30)


## 4. Save Selected Features & Transformed Data

In [5]:
# Save feature list
pd.DataFrame({"feature": selected_features}).to_csv(
    config.TABLES_DIR / "selected_features_final.csv", index=False
)

# Save selected train/test data
save_dataframe(X_train_selected, "X_train_selected.csv")
save_dataframe(X_test_selected, "X_test_selected.csv")

logger.info("Feature selection artifacts saved")
logger.info("Selected features saved to: %s", config.TABLES_DIR / "selected_features_final.csv")
logger.info("Done: 02_Feature_Selection_PSO")

2026-08-26 02:28:02 | INFO     | prostate_bcr | Saved 343 rows → D:\Prostate_BCR\core\data\processed\X_train_selected.csv
2026-08-26 02:28:02 | INFO     | prostate_bcr | Saved 86 rows → D:\Prostate_BCR\core\data\processed\X_test_selected.csv
2026-08-26 02:28:02 | INFO     | prostate_bcr | Feature selection artifacts saved
2026-08-26 02:28:02 | INFO     | prostate_bcr | Selected features saved to: D:\Prostate_BCR\core\outputs\tables\selected_features_final.csv
2026-08-26 02:28:02 | INFO     | prostate_bcr | Done: 02_Feature_Selection_PSO
